# Analyze a GitHub repository

In [1]:
import os
from dotenv import load_dotenv
from github import Github
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate

In [ ]:
# Load environment variables
load_dotenv()

# Configure API keys securely
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Initialize GitHub and LLM clients
github_client = Github(GITHUB_TOKEN)
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.2, openai_api_key=OPENAI_API_KEY)


def analyze_repository(repo_name):
    """
    Analyze a GitHub repository and provide insights.
    """
    repo = github_client.get_repo(repo_name)
    
    description = repo.description or "No description provided"
    stars = repo.stargazers_count
    forks = repo.forks_count
    languages = repo.get_languages()

    try:
        readme = repo.get_readme()
        readme_content = readme.decoded_content.decode('utf-8')
        if len(readme_content) > 3000:
            readme_content = readme_content[:3000] + "... (truncated)"
    except:
        readme_content = "No README found"
    
    template = """
    You are a GitHub repository analyzer. Based on the following information about a GitHub repository,
    provide an analysis of the project's purpose, structure, and potential applications.

    Repository: {repo_name}
    Description: {description}
    Stars: {stars}
    Forks: {forks}
    Languages: {languages}

    README excerpt:
    ```
    {readme}
    ```

    Please provide:
    1. A concise project summary
    2. Main features and capabilities
    3. Technical stack and architecture
    4. Potential use cases
    5. Recommendations for users interested in this project
    """

    prompt = ChatPromptTemplate.from_template(template)
    chain = prompt | llm

    lang_str = ", ".join([f"{lang}: {bytes_count}" for lang, bytes_count in languages.items()])

    response = chain.invoke({
        "repo_name": repo_name,
        "description": description,
        "stars": stars,
        "forks": forks,
        "languages": lang_str,
        "readme": readme_content
    })

    return response.content


def analyze_code_file(repo_name, file_path):
    """
    Analyze a specific code file in a GitHub repository.
    """
    repo = github_client.get_repo(repo_name)

    try:
        file_content = repo.get_contents(file_path)
        content = file_content.decoded_content.decode('utf-8')
        if len(content) > 5000:
            content = content[:5000] + "... (truncated)"
    except Exception as e:
        return f"Error retrieving file: {str(e)}"

    template = """
    You are a code analyst. Analyze the following code file from a GitHub repository:

    Repository: {repo_name}
    File Path: {file_path}

    ```
    {code_content}
    ```

    Please provide:
    1. A summary of what this code does
    2. Key functions/classes and their purposes
    3. Code quality assessment
    4. Potential improvements
    5. Any security or performance concerns
    """

    prompt = ChatPromptTemplate.from_template(template)
    chain = prompt | llm

    response = chain.invoke({
        "repo_name": repo_name,
        "file_path": file_path,
        "code_content": content
    })

    return response.content


def analyze_issues(repo_name, issue_count=5):
    """
    Analyze recent issues in a GitHub repository.
    """
    repo = github_client.get_repo(repo_name)
    issues = repo.get_issues(state="open")

    issues_data = []
    for i, issue in enumerate(issues):
        if i >= issue_count:
            break
        issues_data.append({
            "number": issue.number,
            "title": issue.title,
            "created_at": issue.created_at.isoformat(),
            "author": issue.user.login,
            "body": issue.body[:500] + "..." if issue.body and len(issue.body) > 500 else (issue.body or "No description")
        })

    if not issues_data:
        return "No open issues found in this repository."

    issues_formatted = ""
    for issue in issues_data:
        issues_formatted += f"Issue #{issue['number']}: {issue['title']}\n"
        issues_formatted += f"Author: {issue['author']}, Created: {issue['created_at']}\n"
        issues_formatted += f"Description: {issue['body']}\n\n"

    template = """
    You are a GitHub issues analyst. Analyze the following recent issues from a GitHub repository:

    Repository: {repo_name}

    Issues:
    {issues}

    Please provide:
    1. A summary of common themes or problems
    2. Categorization of issues (bugs, feature requests, documentation, etc.)
    3. Recommendations for prioritizing these issues
    4. Suggestions for potential solutions to common problems
    """

    prompt = ChatPromptTemplate.from_template(template)
    chain = prompt | llm

    response = chain.invoke({
        "repo_name": repo_name,
        "issues": issues_formatted
    })

    return response.content


# Example usage
if __name__ == "__main__":
    target_repo = "langchain-ai/langchain"
    
    print("Repository Analysis:")
    print("-" * 50)
    print(analyze_repository(target_repo))
    print("\n")

    print("Code File Analysis:")
    print("-" * 50)
    print(analyze_code_file(target_repo, "libs/langchain/langchain/chains/base.py"))
    print("\n")

    print("Issues Analysis:")
    print("-" * 50)
    print(analyze_issues(target_repo))